# Factor Analysis

Wiki reference for [factor analysis](https://ml-viz-ruby.vercel.app/wiki/factor-analysis).

**The idea in one sentence.** Factor analysis models data as a few latent factors plus
**per-dimension (diagonal) noise** $\Psi$ — which is exactly what distinguishes it from PCA:
FA can absorb **heteroscedastic** noise that differs across dimensions, so it recovers the true
covariance structure where PCA's spherical-noise assumption fails.

We implement FA via EM from scratch, **validate that it recovers the per-dimension noise and
beats PCA on covariance recovery**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748',
    'grid.color': '#2d3748',
    'axes.grid': True,
})

np.random.seed(42)

## 1. Generating data from the Factor Analysis model

We generate synthetic high-dimensional data from the FA model:
$$z \sim \mathcal{N}(0, I_k), \quad x|z \sim \mathcal{N}(\mu + \Lambda z, \Psi)$$

In [ ]:
n, p, k = 500, 10, 2  # samples, dimensions, factors

# True parameters
Lambda_true = np.random.randn(p, k) * 1.5
mu_true = np.random.randn(p)
psi_true = np.random.uniform(0.1, 2.0, p)  # heteroscedastic noise (different per dim)

# Generate data
Z = np.random.randn(n, k)          # latent factors
noise = np.random.randn(n, p) * np.sqrt(psi_true)
X = Z @ Lambda_true.T + mu_true + noise  # (n, p)

print(f"Data shape: {X.shape}")
print(f"True noise variances (Ψ diag): {psi_true.round(3)}")
print(f"Note: dimensions have DIFFERENT noise levels — PCA can't model this.")

# Show variance per dimension
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(p), X.var(axis=0), color='#6366f1', alpha=0.7, label='Observed variance')
ax.bar(range(p), psi_true, color='#f43f5e', alpha=0.7, label='True noise variance Ψ')
ax.set_xlabel('Dimension')
ax.set_ylabel('Variance')
ax.set_title('Observed Variance vs True Noise Variance per Dimension', color='#e2e8f0')
ax.legend()
plt.tight_layout()
plt.show()

## 2. EM for Factor Analysis

In [ ]:
class FactorAnalysis:
    def __init__(self, n_factors):
        self.k = n_factors

    def fit(self, X, n_iters=100):
        n, p = X.shape
        k = self.k

        # Initialize
        self.mu = X.mean(axis=0)
        self.Lambda = np.random.randn(p, k) * 0.1
        self.Psi = np.ones(p)  # diagonal noise

        X_centered = X - self.mu
        self.ll_history = []

        for it in range(n_iters):
            Psi_inv = 1.0 / self.Psi  # (p,)
            L = self.Lambda

            # E-step: posterior of z given x
            # Sigma_z = (I + LᵀΨ⁻¹L)⁻¹
            M = np.eye(k) + (L.T * Psi_inv) @ L
            M_inv = np.linalg.inv(M)

            # E[z|x] = M_inv @ Lᵀ @ Ψ⁻¹ @ x  (n, k)
            Ez = X_centered @ (Psi_inv[:, None] * L) @ M_inv  # (n, k)
            Ezz = n * M_inv + Ez.T @ Ez   # E[zzᵀ] summed over all examples

            # M-step: update Lambda and Psi
            # Lambda_new = (Σ x Ezᵀ)(Σ Ezzᵀ)⁻¹
            self.Lambda = (X_centered.T @ Ez) @ np.linalg.inv(Ezz)
            # Psi_new = diag( (1/n) Σ (xᵢ - μ)² - Lambda Eᵢ[z] )
            recon = Ez @ self.Lambda.T
            self.Psi = np.maximum((X_centered ** 2).mean(0) - (recon * X_centered).mean(0), 1e-6)

        return self

    @property
    def covariance(self):
        return self.Lambda @ self.Lambda.T + np.diag(self.Psi)


fa = FactorAnalysis(n_factors=k)
fa.fit(X)

print("True Ψ:    ", psi_true.round(3))
print("Fitted Ψ:  ", fa.Psi.round(3))

# Compare recovered vs true noise — should be close
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(np.arange(p) - 0.2, psi_true, width=0.4, color='#10b981', alpha=0.8, label='True Ψ')
ax.bar(np.arange(p) + 0.2, fa.Psi,   width=0.4, color='#f59e0b', alpha=0.8, label='FA fitted Ψ')
ax.set_xlabel('Dimension')
ax.set_ylabel('Noise variance')
ax.set_title('EM Recovers Per-Dimension Noise Variances', color='#e2e8f0')
ax.legend()
plt.tight_layout()
plt.show()

### Validate: FA recovers the per-dimension noise variances

The data was generated with a *different* noise variance per dimension ($\Psi$ diagonal,
heteroscedastic). FA's whole point is to estimate that diagonal — so the fitted $\Psi$ should
strongly correlate with the true one. We confirm.

In [ ]:
corr = np.corrcoef(fa.Psi, psi_true)[0, 1]
print(f'correlation between fitted Psi and true Psi: {corr:.3f}')
assert corr > 0.9, 'FA recovers the per-dimension (heteroscedastic) noise variances'
print('\n✅ FA separates shared factor structure from per-dimension noise')

## 3. FA vs PCA — handling heteroscedastic noise

PCA assumes equal noise across all dimensions. FA models per-dimension noise. We compare their covariance reconstruction on data with very unequal noise levels.

In [ ]:
from numpy.linalg import svd

# PCA approximation of the covariance
X_c = X - X.mean(0)
U, S, Vt = svd(X_c, full_matrices=False)
Lambda_pca = Vt[:k].T * (S[:k] / np.sqrt(len(X)))  # (p, k)
sigma2_pca = ((S[k:]**2).sum() / ((len(X)) * (p - k)))  # spherical noise estimate
Sigma_pca = Lambda_pca @ Lambda_pca.T + sigma2_pca * np.eye(p)

Sigma_true = Lambda_true @ Lambda_true.T + np.diag(psi_true)
Sigma_fa   = fa.Lambda @ fa.Lambda.T + np.diag(fa.Psi)
Sigma_emp  = np.cov(X.T)

def frob_error(A, B): return np.linalg.norm(A - B, 'fro')

print(f"Frobenius error (true Σ vs empirical): {frob_error(Sigma_true, Sigma_emp):.3f}")
print(f"Frobenius error (true Σ vs FA):        {frob_error(Sigma_true, Sigma_fa):.3f}")
print(f"Frobenius error (true Σ vs PPCA):      {frob_error(Sigma_true, Sigma_pca):.3f}")
print()
print("FA wins when noise is heteroscedastic (different per dimension).")

### Validate: FA beats PCA on covariance recovery

PCA (probabilistic PCA) assumes a single **spherical** noise level, so when the true noise is
heteroscedastic it mis-fits the covariance. FA's diagonal $\Psi$ fits it far better. We confirm
FA's Frobenius error to the true covariance is smaller than PCA's.

In [ ]:
err_fa = frob_error(Sigma_true, Sigma_fa)
err_pca = frob_error(Sigma_true, Sigma_pca)
print(f'Frobenius error to true covariance: FA={err_fa:.3f}  PCA={err_pca:.3f}')
assert err_fa < err_pca, 'FA models per-dimension noise -> recovers covariance better than spherical PCA'
print('\n✅ heteroscedastic noise is where FA wins over PCA')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **rotational indeterminacy** | factor orientation is arbitrary (demo) — rotate to interpret |
| **choosing #factors** | too few underfit, too many overfit; use the elbow / held-out |
| **PCA vs FA** | PCA assumes spherical noise; FA models per-dimension (verified) |
| **Heywood cases** | a noise variance can go negative/zero — constrain $\Psi$ |
| **non-Gaussian data** | FA's Gaussian assumption may not hold |

Demo: Lambda and Lambda@R give the same covariance — factors have no fixed orientation.

In [ ]:
# FA's identifiability gotcha: the factor loadings Lambda are only defined UP TO A ROTATION.
# For any orthogonal matrix R, Lambda and Lambda@R produce the EXACT same covariance
# (Lambda R R^T Lambda^T = Lambda Lambda^T), so the 'factors' have no inherent orientation —
# you cannot read individual factor meanings without an extra rotation criterion (e.g. varimax).
L = fa.Lambda
theta = 0.7
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
Sigma_L = L @ L.T
Sigma_LR = (L @ R) @ (L @ R).T
print(f'max |Sigma(Lambda) - Sigma(Lambda R)| = {np.abs(Sigma_L - Sigma_LR).max():.2e}')
assert np.allclose(Sigma_L, Sigma_LR), 'Lambda and Lambda@R give the identical covariance -> rotational indeterminacy'
print('\nFA loadings are identified only up to rotation -> use varimax/oblique rotation to interpret factors.')

## ✏️ Your turn

**Exercise 1 — Choose number of factors.** Fit Factor Analysis with k ∈ {1, 2, 3, 4, 5} factors. Compute the Frobenius reconstruction error for each k and plot it. At what k does the error stop improving significantly?

In [ ]:
k_values = [1, 2, 3, 4, 5]
# TODO(you): fit FactorAnalysis for each k and compute frob_error(Sigma_true, Sigma_fa)
# errors = [frob_error(Sigma_true, FactorAnalysis(k).fit(X).covariance) for k in k_values]
# plt.plot(k_values, errors, 'o-', ...)

In [ ]:
# Assert cell
errors_ref = [frob_error(Sigma_true, FactorAnalysis(kk).fit(X).covariance) for kk in k_values]
print("Errors by k:", [f"{e:.3f}" for e in errors_ref])
assert errors_ref[0] > errors_ref[1], "More factors should reduce reconstruction error"
assert errors_ref[k_values.index(k)] < errors_ref[0], "True k factors should do well"

<details><summary>Solution</summary>

```python
errors = [frob_error(Sigma_true, FactorAnalysis(kk).fit(X).covariance) for kk in k_values]
plt.figure(figsize=(7, 4))
plt.plot(k_values, errors, 'o-', color='#6366f1', linewidth=2)
plt.axvline(k, color='#f43f5e', linestyle='--', label=f'True k={k}')
plt.xlabel('Number of factors k')
plt.ylabel('Frobenius error (vs true Σ)')
plt.title('Choosing k in Factor Analysis')
plt.legend()
plt.show()
```

The error drops sharply up to the true number of factors (k=2) and then flattens or improves only marginally. This elbow in the error curve is a reliable heuristic for choosing k — similar to the scree plot in PCA.

</details>

## Key takeaways

- **FA = factors + diagonal noise $\Psi$:** it models per-dimension (heteroscedastic) noise,
  unlike PCA (verified).
- **Recovers $\Psi$** and the covariance structure better than PCA when noise varies per
  dimension (verified).
- **Rotational indeterminacy:** loadings are defined only up to rotation (demo) — use varimax to
  interpret.
- **PCA is FA's special case** with isotropic noise $\Psi = \sigma^2 I$.